# 06. Kiểm định Mardia và phân tích độ bền (sensitivity)
Mục đích: (1) kiểm tra chuẩn đa biến bằng Mardia (bổ sung cho Henze-Zirkler); (2) kiểm tra kết luận MANOVA có thay đổi theo cách xử lý dữ liệu hay không.

In [1]:
from pathlib import Path
BASE_DIR = Path.cwd().parent if Path.cwd().name in ("src", "notebooks") else Path.cwd()
import numpy as np, pandas as pd
from scipy.stats import chi2, norm
import pingouin as pg
import warnings; warnings.filterwarnings("ignore")
from statsmodels.multivariate.manova import MANOVA
from scipy.stats import chi2 as _chi2
RAW = BASE_DIR / "data/raw/arabica_data_cleaned.csv"
DV = ["Fragrance","Flavor","Aftertaste","Acidity","Body","Moisture"]
print("Thư mục gốc:", BASE_DIR)

Thư mục gốc: f:\PTTKNC\Nhom_09_Detai12


## 1. Kiểm định Mardia (skewness và kurtosis)
H0: dữ liệu có phân phối chuẩn đa biến. Áp dụng cho từng nhóm Origin trên dữ liệu đã làm sạch.

In [2]:
def mardia(X):
    X = np.asarray(X, float); n, p = X.shape
    Xc = X - X.mean(axis=0)
    S = Xc.T @ Xc / n                      # ước lượng MLE (chia n)
    Si = np.linalg.inv(S)
    D = Xc @ Si @ Xc.T
    b1 = (D**3).sum() / n**2               # skewness
    b2 = (np.diag(D)**2).mean()            # kurtosis
    skew_stat = n * b1 / 6
    skew_df = p*(p+1)*(p+2)/6
    skew_p = 1 - chi2.cdf(skew_stat, skew_df)
    kurt_z = (b2 - p*(p+2)) / np.sqrt(8*p*(p+2)/n)
    kurt_p = 2*(1 - norm.cdf(abs(kurt_z)))
    return dict(n=n, b1p=b1, skew_p=skew_p, b2p=b2, kurt_z=kurt_z, kurt_p=kurt_p)

clean = pd.read_excel(BASE_DIR / "data/processed/clean_data.xlsx")
rows = []
r = mardia(clean[DV].to_numpy(float)); r["Nhóm"] = "Toàn bộ"; rows.append(r)
for g, sub in clean.groupby("Origin"):
    r = mardia(sub[DV].to_numpy(float)); r["Nhóm"] = g; rows.append(r)
mardia_df = pd.DataFrame(rows)[["Nhóm","n","b1p","skew_p","b2p","kurt_z","kurt_p"]]
print("Kiểm định Mardia (bổ sung cho Henze-Zirkler):")
print(mardia_df.round(4).to_string(index=False))
print("\nKết luận: p < 0.05 nghĩa là bác bỏ chuẩn đa biến; phù hợp với kết quả Henze-Zirkler ở notebook 02.")

Kiểm định Mardia (bổ sung cho Henze-Zirkler):


     Nhóm   n     b1p  skew_p     b2p  kurt_z  kurt_p
  Toàn bộ 720  3.3600  0.0000 57.7458 13.3450  0.0000
   Brazil 129  6.9671  0.0000 60.1732  7.0556  0.0000
 Colombia 180  3.3426  0.0003 53.0817  3.4792  0.0005
Guatemala 175 14.2330  0.0000 70.1887 14.9791  0.0000
   Mexico 236 11.1617  0.0000 65.0092 13.3344  0.0000

Kết luận: p < 0.05 nghĩa là bác bỏ chuẩn đa biến; phù hợp với kết quả Henze-Zirkler ở notebook 02.


## 2. Phân tích độ bền: MANOVA theo các kịch bản xử lý dữ liệu
- **A. Giữ Moisture = 0** (cách làm ban đầu của nhóm)
- **B. Moisture = 0 là missing** (kết quả chính của đề tài)
- **C. Bỏ hẳn biến Moisture** (p = 5), để kiểm tra kết luận không phụ thuộc vào biến này

In [3]:
def prep(zero_as_missing=True, drop_moisture=False):
    d = pd.read_csv(RAW)[["Aroma","Flavor","Aftertaste","Acidity","Body","Moisture","Country.of.Origin"]]
    d = d.rename(columns={"Aroma":"Fragrance","Country.of.Origin":"Origin"})
    d = d[d["Origin"].isin(["Mexico","Colombia","Guatemala","Brazil"])].copy()
    dv = DV[:-1] if drop_moisture else DV
    for c in DV: d[c] = pd.to_numeric(d[c], errors="coerce")
    if zero_as_missing: d.loc[d["Moisture"] == 0, "Moisture"] = np.nan
    d = d.dropna(subset=dv + ["Origin"]).reset_index(drop=True)
    X = d[dv].to_numpy(float)
    D2 = np.einsum("ij,jk,ik->i", X - X.mean(0), np.linalg.inv(np.cov(X, rowvar=False)), X - X.mean(0))
    d = d[~(D2 > chi2.ppf(0.999, len(dv)))].reset_index(drop=True)
    return d, dv

scen = {"A. Giữ Moisture = 0": prep(False, False),
        "B. Moisture = 0 là missing (chính)": prep(True, False),
        "C. Bỏ biến Moisture": prep(True, True)}
out = []
for name, (d, dv) in scen.items():
    bm = pg.box_m(data=d, dvs=dv, group="Origin")["pval"].iloc[0]
    st = MANOVA.from_formula(" + ".join(dv) + " ~ Origin", data=d).mv_test().results["Origin"]["stat"]
    out.append({"Kịch bản": name, "n": len(d), "Số biến p": len(dv), "Box M p": bm,
                "Wilks": st.loc["Wilks' lambda","Value"], "Wilks p": st.loc["Wilks' lambda","Pr > F"],
                "Pillai": st.loc["Pillai's trace","Value"], "Pillai p": st.loc["Pillai's trace","Pr > F"]})
sens_df = pd.DataFrame(out)
print(sens_df.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nNhận xét: Origin có ảnh hưởng có ý nghĩa ở cả ba kịch bản, nên kết luận không phụ thuộc vào cách xử lý Moisture.")
print("Riêng kịch bản A, Moisture = 0 làm tăng mức khác biệt Moisture giữa các vùng một cách giả tạo.")

                          Kịch bản   n  Số biến p   Box M p  Wilks   Wilks p  Pillai  Pillai p
               A. Giữ Moisture = 0 720          6 7.216e-47 0.5961  2.21e-67  0.4467  1.46e-62
B. Moisture = 0 là missing (chính) 582          6 3.026e-30 0.5824 2.168e-55  0.4767 5.096e-53
               C. Bỏ biến Moisture 721          5 1.142e-19 0.6968 2.233e-46  0.3312 5.697e-45

Nhận xét: Origin có ảnh hưởng có ý nghĩa ở cả ba kịch bản, nên kết luận không phụ thuộc vào cách xử lý Moisture.
Riêng kịch bản A, Moisture = 0 làm tăng mức khác biệt Moisture giữa các vùng một cách giả tạo.


In [4]:
out_file = BASE_DIR / "data/processed/sensitivity_results.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as w:
    mardia_df.to_excel(w, sheet_name="Mardia", index=False)
    sens_df.to_excel(w, sheet_name="Sensitivity", index=False)
print("Đã lưu:", out_file)

Đã lưu: f:\PTTKNC\Nhom_09_Detai12\data\processed\sensitivity_results.xlsx
